In [ ]:
include( "../geom.jl" )
include( "../kernel.jl" )

include( "../0-params/rho-list.jl" )

using ThreadsX

_, base = expfolder( "system" )

runsim = false;

In [ ]:
# Hyper parameters.
N = 128;  s = 0.251
N̂ = 128

# Extract critical noise.
folderlist = [findfolder( N, ρlist[i], s; base=base, crit=true ) for i ∈ 1:Nρ]
f̂olderlist = [findfolder( N̂, ρlist[i], s; base=base, crit=true ) for i ∈ 1:Nρ]

# Unpack the kernel parameters.
γlist = vec.( readdlm.( folderlist.*"kernel-parameters.txt" ) )
println( "Kernel parameters: ", γlist )

In [ ]:
# Import area and perimeter scaling values.
βlist = first.( vec.( readdlm.( folderlist.*"area-perimeter-scaling.txt" ) ), 2 )

# Import the cluster change frequency values.
θlist = vec( first.( readdlm.( folderlist.*"optimal-rates-hat.txt" )', 3 ) )

# Extract the power law.
λlist = first.( readdlm.( folderlist.*"power-law.txt" ) );

In [ ]:
# Import cluster mass distribution extracted from data.
Flist = map.( F -> round.( defInt, F ), vec.( readdlm.( f̂olderlist.*"cluster-frequency.txt" ) ) )
flist = [F./sum( F ) for F ∈ Flist]

# Unpack the probability map data.
ℓlist = [polyloglaw.( 1:N̂; β=β ) for β ∈ βlist]
Alist = [(ℓ.^2)./(4π) for ℓ ∈ ℓlist]

# Construct the probability map.
Ωlist = ThreadsX.map( θ -> [P( j; N=N̂, i=i, λℓ=θ[1], λr=θ[2], κ=θ[3] ) for i in 1:N̂, j in 1:N̂], θlist );

In [ ]:
# Simulate the master equations in the larger system size..
T = 1.0
f̂list = [kernelloss( Flist[i], γlist[i], (N̂, Ωlist[i], Alist[i], ℓlist[i]); T=T, verbose=true )[2]
    for i ∈ 1:Nρ];

In [ ]:
# Compute the log-likelihood of the steady state distributions.
λ̂list = [10.0^optimize( createzipfloss( N̂, N̂.*f̂list[i] ), [log10( 1.5 )] ).minimizer[1] for i ∈ 1:Nρ];

In [ ]:
# Select density to illustrate.
i = 21
f = flist[i]
f̂ = f̂list[i]

# Plot bounds.
xmax = ceil( log10( N̂ ) )
ymin = floor( minimum( log10.( f[f .> 0] ) ) )

# Plot the final distribution from the mean-field equations.
plt = plot(; size=(350,200), legend=:bottomleft )

# Population size.
plot!( plt, [N̂, N̂], 10.0.^[ymin,0]; color=:black, linestyle=:dot, lw=2, label="system size" )

# Plot mean-field and simulation cluster mass distributions.
scatter!( plt, f; color=:gray, label="raw data" )
scatter!( plt, f̂; color=:black, markersize=3, lw=2, label="master eq." )

plot!( plt; xlims=10.0.^[0,xmax], xscale=:log10 )
plot!( plt; ylims=10.0.^[ymin,0], yscale=:log10 )

plot!( plt; xlabel="cluster size, "*L"i", ylabel="probability, "*L"P(i)", legend=:bottomleft )
# plot!( plt; title=latexstring( "∣ λ - \\hat \\lambda ∣ = $(round( abs( λ - λ̂ ), digits=3 ))" ), titleposition=:left )

# saveplot( plt, "../../figures/extrapolated-cmd_N-$(N)"; dpi=600 )

In [ ]:
# Save the simulated distribution.
if true
    for i ∈ 1:Nρ
        writedlm( folderlist[i]*"cluster-distribution-hat.txt", f̂list[i] )
    end
end